# 5 — BankNifty Enhanced Charts V2 (Split Figures + ROC AVG Line)

**Run frequency:** During market hours alongside Notebook 2

**What this does:**
1. Logs in to Zerodha KiteConnect (needed for live ATM strike)
2. Starts a SparkSession
3. Reads instruments + expiries from Parquet
4. Builds the options DataFrame for the configured ATM strike
5. Renders **enhanced V2** Plotly charts:

   | Figure | Layout | Content |
   |--------|--------|---------|
   | 1 | 1×2 | **Straddle Price & Signals** \| **ROC AVG + ROC Average Line** |
   | 2 | 1×2 | **CE vs PE Close** \| **CE/PE ROC & Ratio** *(straddle only)* |
   | 3 | 1×2 | Combined OI \| CE/PE OI + PUT−CALL OI secondary Y |
   | 4 | standalone | PCR |
   | 5 | 2×1 shared-x | Straddle Price/VWAP/OIWAP + PUT−CALL OI indicator pane |
   | 6 | 2×1 shared-x | OHLC Candlestick + Volume |

   All charts use `DD-Mon HH:MM` time labels with vertical day-boundary lines.

6. **Auto-refreshes** every minute; stops automatically at `END_HOUR:END_MINUTE`

---
### ⚙️ Configuration

| Parameter | Default | Description |
|-----------|---------|-------------|
| `STRIKE_LEVEL_NAME` | `'ATM'` | Which strike to chart |
| `CE_OR_PE` | `'E'` | `'CE'`, `'PE'`, or `'E'` for straddle |
| `NUM_DAYS` | `2` | Days of data to display |
| `IS_LATEST_DAY` | `True` | Show only today's data |
| `CUSTOM_STRIKE` | `0` | Override ATM (0 = live LTP) |
| `NUM_STRIKES` | `11` | Strikes each side of ATM to load |
| `LOOP_INTERVAL_MIN` | `1` | Chart refresh frequency (minutes) |
| `END_HOUR` | `15` | Stop loop at this hour (24h, set `None` to run indefinitely) |
| `END_MINUTE` | `30` | Stop loop at this minute |

In [ ]:
# ── Setup ──────────────────────────────────────────────────────────────────
import sys, os
# sys.path.insert(0, os.path.abspath('..'))
# os.chdir('..')  # Run from repo root so DataFiles/ paths resolve correctly

In [ ]:
# ── CONFIG — edit these values ─────────────────────────────────────────────
STRIKE_LEVEL_NAME  = 'ATM'   # 'ATM', 'ATM+1', 'ATM-2', etc.
CE_OR_PE           = 'E'     # 'CE', 'PE', or 'E' for straddle
NUM_DAYS           = 2       # Days of OHLC data to show
IS_LATEST_DAY      = True    # True = show today only; False = show NUM_DAYS
NUM_DAYS_BACK      = 0       # 0 = latest, 1 = go back 1 day
CUSTOM_STRIKE      = 0       # 0 = use live LTP; override e.g. 56500
NUM_STRIKES        = 11      # Strikes each side of ATM to load
LOOP_INTERVAL_MIN  = 1       # Chart refresh frequency (minutes)
INTERVAL           = '3minute'

# Time-based loop exit — set to None to run indefinitely
END_HOUR           = 15      # 24-hour format (e.g. 15 = 3 PM)
END_MINUTE         = 30      # Minute (e.g. 30 = :30)

In [ ]:
# ── Step 1: Login ──────────────────────────────────────────────────────────
from utils.kite_helpers import kite_login, get_spark_session

kite, kws, access_token = kite_login()
print('✅ Login successful')

In [ ]:
# ── Step 2: Start SparkSession ─────────────────────────────────────────────
spark = get_spark_session(app_name='5_Enhanced_Charts_BNF')
print('✅ Spark session ready')

In [ ]:
# ── Step 3: Load instruments and expiry data ───────────────────────────────
from utils.strike_utils import read_instruments, get_expiry_dates, get_Options_DF, get_ATM_Strike
from utils.strike_utils import BANKNIFTY_INDEX_TOKEN

bnf_options, bnf_futures, nifty_options, nifty_futures, expiries_df = read_instruments(spark)
bnf_expiries = get_expiry_dates(expiries_df, 'BANKNIFTY')

print(f"BankNifty current week expiry: {bnf_expiries['current_week']}")
print(f"BankNifty next week expiry:    {bnf_expiries['next_week']}")

In [ ]:
# ── Step 4: Build options DataFrame for ATM region ────────────────────────
if CUSTOM_STRIKE > 0:
    bnf_atm = CUSTOM_STRIKE
else:
    bnf_atm = get_ATM_Strike(kite, BANKNIFTY_INDEX_TOKEN, rounding_number=100)

print(f'BankNifty ATM Strike = {bnf_atm}')

Banknifty_Options_DF = get_Options_DF(
    spark                = spark,
    options_df_from_file = bnf_options,
    atm_strike           = bnf_atm,
    current_expiry       = bnf_expiries['current_week'],
    strike_range         = 100,
    num_strikes          = NUM_STRIKES,
)
print(f"Options loaded: {Banknifty_Options_DF.count()} rows")
Banknifty_Options_DF.show(5)

In [ ]:
# ── Step 5: Fetch data and compute analytics ───────────────────────────────
# This cell fetches the raw data and runs the analytics pipeline.
# Run this cell once, then run the chart cells below to render each figure.
from utils.chart_utils import compute_analytics, get_Strike_OHLC_data
from utils.enhanced_charts_v2 import _add_extra_analytics, _build_x_axis, _apply_dark_layout, _add_day_boundaries, COLORS
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from IPython.display import clear_output

is_straddle = CE_OR_PE not in ('CE', 'PE')
cols_single   = ['date', 'open', 'high', 'low', 'close', 'oi', 'strike', 'day_num']
cols_straddle = ['date', 'open', 'high', 'low', 'close',
                 'CE_close', 'PE_close', 'oi', 'CE_oi', 'PE_oi', 'strike', 'day_num']
select_cols = cols_straddle if is_straddle else cols_single

raw_df = get_Strike_OHLC_data(
    spark                = spark,
    options_df           = Banknifty_Options_DF,
    name                 = 'BANKNIFTY',
    expiry               = bnf_expiries['current_week'],
    strike_level_name    = STRIKE_LEVEL_NAME,
    ce_or_pe             = CE_OR_PE,
    interval             = INTERVAL,
    num_days             = NUM_DAYS,
    num_days_back        = NUM_DAYS_BACK,
).select(select_cols)

df_pd   = compute_analytics(spark, raw_df, CE_OR_PE, IS_LATEST_DAY)
strike  = df_pd.iloc[0]['strike']
df      = _add_extra_analytics(df_pd)
x, boundary_x = _build_x_axis(df)

# ROC AVG average line — expanding cumulative mean (mirrors Straddle AVG logic)
roc_avg_line = df['ROC_AVG_close'].expanding().mean()

from datetime import datetime
now_str    = datetime.now().strftime('%Y-%m-%d %I:%M %p')
title_base = f'Strike: ({STRIKE_LEVEL_NAME}) / {strike}  —  {now_str}'

print(f'✅ Data ready — {len(df)} rows | Strike: {strike}')

In [ ]:
# ── Figure 1: Straddle Price & Signals  |  ROC AVG + ROC Average Line ─────
fig1 = make_subplots(
    rows=1, cols=2,
    subplot_titles=['Straddle Price & Signals', 'ROC AVG'],
    horizontal_spacing=0.06,
)

# ── Col 1: Straddle Price & Signals ────────────────────────────────────────
# Bollinger band fill
fig1.add_trace(go.Scatter(x=x, y=df['bb_upper'], mode='lines', line=dict(width=0),
    showlegend=False, hoverinfo='skip'), row=1, col=1)
fig1.add_trace(go.Scatter(x=x, y=df['bb_lower'], mode='lines', line=dict(width=0),
    fill='tonexty', fillcolor=COLORS['bb_fill'], showlegend=False, hoverinfo='skip'), row=1, col=1)
fig1.add_trace(go.Scatter(x=x, y=df['bb_upper'], mode='lines', name='BB Upper',
    line=dict(color=COLORS['bb_line'], width=1, dash='dot')), row=1, col=1)
fig1.add_trace(go.Scatter(x=x, y=df['bb_lower'], mode='lines', name='BB Lower',
    line=dict(color=COLORS['bb_line'], width=1, dash='dot')), row=1, col=1)
fig1.add_trace(go.Scatter(x=x, y=df['close'], mode='lines', name='Straddle Close',
    line=dict(color=COLORS['straddle_close'], width=2.5)), row=1, col=1)
trailing_sl = (df['incremental_min_close'] + df['incremental_close_avg']) / 2
fig1.add_trace(go.Scatter(x=x, y=trailing_sl, mode='lines', name='Trailing SL',
    line=dict(color=COLORS['trailing_sl'], width=2, dash='dash')), row=1, col=1)
if 'StopLoss' in df.columns:
    fig1.add_trace(go.Scatter(x=x, y=df['StopLoss'], mode='lines', name='StopLoss',
        line=dict(color=COLORS['stoploss'], width=2)), row=1, col=1)
fig1.add_trace(go.Scatter(x=x, y=df['incremental_close_avg'], mode='lines', name='Straddle AVG',
    line=dict(color=COLORS['avg'], width=1.5, dash='dot')), row=1, col=1)
if 'EntryAllowed' in df.columns:
    fig1.add_trace(go.Scatter(x=x, y=df['EntryAllowed'], mode='markers+lines', name='Entry Allowed',
        line=dict(color=COLORS['entry_allowed'], width=2),
        marker=dict(size=4, color=COLORS['entry_allowed'])), row=1, col=1)

# ── Col 2: ROC AVG + ROC Average Line ──────────────────────────────────────
fig1.add_trace(go.Scatter(x=x, y=[0]*len(df), mode='lines',
    line=dict(color=COLORS['midline'], width=1), showlegend=False), row=1, col=2)
fig1.add_trace(go.Scatter(x=x, y=df['ROC_AVG_close'], mode='lines', name='ROC AVG',
    line=dict(color=COLORS['roc_avg'], width=2)), row=1, col=2)
# ROC Average Line — expanding cumulative mean, same style as Straddle AVG
fig1.add_trace(go.Scatter(x=x, y=roc_avg_line, mode='lines', name='ROC Average',
    line=dict(color=COLORS['avg'], width=1.5, dash='dot')), row=1, col=2)

_add_day_boundaries(fig1, boundary_x, [1, 1], [1, 2])
_apply_dark_layout(fig1, f'Price & Signals — {title_base}', height=500, width=1400)
fig1.show()

In [ ]:
# ── Figure 2: CE vs PE Close  |  CE/PE ROC & Ratio  (straddle only) ───────
if is_straddle:
    fig2 = make_subplots(
        rows=1, cols=2,
        subplot_titles=['CE vs PE Close', 'CE / PE ROC & Ratio'],
        horizontal_spacing=0.06,
    )

    # ── Col 1: CE vs PE close ───────────────────────────────────────────────
    fig2.add_trace(go.Scatter(x=x, y=df['CE_close'], mode='lines', name='CALL',
        line=dict(color=COLORS['ce_close'], width=2)), row=1, col=1)
    fig2.add_trace(go.Scatter(x=x, y=df['PE_close'], mode='lines', name='PUT',
        line=dict(color=COLORS['pe_close'], width=2)), row=1, col=1)
    fig2.add_trace(go.Scatter(x=x, y=df['incremental_CE_close_avg'], mode='lines', name='CE AVG',
        line=dict(color=COLORS['ce_avg'], width=1, dash='dot')), row=1, col=1)
    fig2.add_trace(go.Scatter(x=x, y=df['incremental_PE_close_avg'], mode='lines', name='PE AVG',
        line=dict(color=COLORS['pe_avg'], width=1, dash='dot')), row=1, col=1)
    if 'Open_Max_Close' in df.columns:
        fig2.add_trace(go.Scatter(x=x, y=df['Open_Max_Close'], mode='lines', name='Open Max Close',
            line=dict(color=COLORS['open_max_close'], width=2, dash='dashdot')), row=1, col=1)

    # ── Col 2: CE/PE ROC + Ratio ────────────────────────────────────────────
    fig2.add_trace(go.Scatter(x=x, y=[0]*len(df), mode='lines',
        line=dict(color=COLORS['midline'], width=1), showlegend=False), row=1, col=2)
    fig2.add_trace(go.Scatter(x=x, y=df['ROC_AVG_CE_close'], mode='lines', name='CE ROC AVG',
        line=dict(color=COLORS['ce_roc'], width=2)), row=1, col=2)
    fig2.add_trace(go.Scatter(x=x, y=df['ROC_AVG_PE_close'], mode='lines', name='PE ROC AVG',
        line=dict(color=COLORS['pe_roc'], width=2)), row=1, col=2)
    if 'ce_pe_ratio' in df.columns:
        fig2.add_trace(go.Scatter(x=x, y=df['ce_pe_ratio'], mode='lines', name='CE/PE Ratio',
            line=dict(color=COLORS['ratio_line'], width=1.5, dash='dash')), row=1, col=2)

    _add_day_boundaries(fig2, boundary_x, [1, 1], [1, 2])
    _apply_dark_layout(fig2, f'CE / PE — {title_base}', height=450, width=1400)
    fig2.show()
else:
    print('ℹ️  CE/PE figure skipped — single-leg mode (CE_OR_PE is CE or PE)')

In [ ]:
# ── Figure 3: Open Interest (PUT-CALL OI diff on secondary Y) ─────────────
from plotly.subplots import make_subplots as _ms
oi_cols  = 2 if is_straddle else 1
oi_titles = (['Combined OI', 'CE / PE OI  (right axis = PUT−CALL OI)']
              if is_straddle else ['Open Interest'])
oi_specs  = ([[{'secondary_y': False}, {'secondary_y': True}]]
              if is_straddle else [[{'secondary_y': False}]])
fig3 = _ms(rows=1, cols=oi_cols, subplot_titles=oi_titles, specs=oi_specs)

fig3.add_trace(go.Scatter(x=x, y=df['oi'], mode='lines', name='Combined OI',
    line=dict(color=COLORS['oi_combined'], width=2)), row=1, col=1)
fig3.add_trace(go.Scatter(x=x, y=df['incremental_oi_avg'], mode='lines', name='OI AVG',
    line=dict(color=COLORS['oi_avg'], width=1.5, dash='dot')), row=1, col=1)

if is_straddle:
    fig3.add_trace(go.Scatter(x=x, y=df['CE_oi'], mode='lines', name='CALL OI',
        line=dict(color=COLORS['ce_oi'], width=2)), row=1, col=2, secondary_y=False)
    fig3.add_trace(go.Scatter(x=x, y=df['PE_oi'], mode='lines', name='PUT OI',
        line=dict(color=COLORS['pe_oi'], width=2)), row=1, col=2, secondary_y=False)
    if 'oi_diff' in df.columns:
        fig3.add_trace(go.Scatter(x=x, y=df['oi_diff'], mode='lines', name='PUT OI − CALL OI',
            line=dict(color=COLORS['oi_diff'], width=1.5, dash='dash')),
            row=1, col=2, secondary_y=True)
        fig3.update_yaxes(title_text='PUT OI − CALL OI', secondary_y=True, row=1, col=2,
            showgrid=False, tickfont=dict(size=9, color=COLORS['oi_diff']))

_add_day_boundaries(fig3, boundary_x, [1]*oi_cols, list(range(1, oi_cols+1)))
_apply_dark_layout(fig3, f'Open Interest — {title_base}', height=400, width=1400)
fig3.show()

In [ ]:
# ── Figure 4: PCR standalone ───────────────────────────────────────────────
if is_straddle and 'oi_pcr' in df.columns:
    fig4 = go.Figure()
    fig4.add_trace(go.Scatter(x=x, y=df['oi_pcr'], mode='lines', name='OI PCR (PE/CE)',
        line=dict(color=COLORS['oi_pcr'], width=2)))
    fig4.add_hline(y=1.0, line=dict(color=COLORS['midline'], width=1.5, dash='dash'),
        annotation_text='PCR = 1', annotation_position='top right',
        annotation_font=dict(color=COLORS['text'], size=10))
    fig4.update_layout(
        height=300, width=1400,
        title=dict(text=f'OI Put-Call Ratio (PCR) — {title_base}',
                   font=dict(size=16, color=COLORS['text']), x=0.5),
        paper_bgcolor=COLORS['paper'], plot_bgcolor=COLORS['bg'],
        font=dict(color=COLORS['text'], size=11),
        legend=dict(bgcolor='rgba(30,30,47,0.85)', bordercolor='#444', borderwidth=1,
                    font=dict(size=10, color=COLORS['text'])),
        hovermode='x unified', margin=dict(l=60, r=30, t=55, b=60),
        yaxis=dict(title='PCR', showgrid=True, gridcolor=COLORS['grid']),
        xaxis=dict(showgrid=True, gridcolor=COLORS['grid'],
                   tickfont=dict(size=9), tickangle=-45, nticks=20),
    )
    for bx in boundary_x:
        fig4.add_vline(x=bx, line=dict(color=COLORS['day_boundary'], width=1.5, dash='dash'))
    fig4.show()

In [ ]:
# ── Figure 5: Straddle Price / VWAP / OIWAP  +  PUT-CALL OI indicator ─────
fig5 = make_subplots(
    rows=2, cols=1, shared_xaxes=True,
    row_heights=[0.70, 0.30], vertical_spacing=0.04,
    subplot_titles=['Straddle Price · VWAP · OIWAP', 'PUT OI − CALL OI'],
)
fig5.add_trace(go.Scatter(x=x, y=df['close'], mode='lines', name='Straddle Price',
    line=dict(color=COLORS['straddle_close'], width=2)), row=1, col=1)
if 'vwap' in df.columns and df['vwap'].notna().any():
    fig5.add_trace(go.Scatter(x=x, y=df['vwap'], mode='lines', name='VWAP',
        line=dict(color=COLORS['vwap'], width=2, dash='dash')), row=1, col=1)
if 'oiwap' in df.columns and df['oiwap'].notna().any():
    fig5.add_trace(go.Scatter(x=x, y=df['oiwap'], mode='lines', name='OIWAP',
        line=dict(color=COLORS['oiwap'], width=2, dash='dot')), row=1, col=1)
if is_straddle and 'oi_diff' in df.columns:
    fig5.add_trace(go.Scatter(x=x, y=df['oi_diff'], mode='lines', name='PUT OI − CALL OI',
        line=dict(color=COLORS['oi_diff'], width=1.5),
        fill='tozeroy', fillcolor='rgba(206,147,216,0.15)'), row=2, col=1)
    fig5.add_hline(y=0, line=dict(color=COLORS['midline'], width=1, dash='dash'), row=2, col=1)
_add_day_boundaries(fig5, boundary_x, [1, 2], [1, 1])
_apply_dark_layout(fig5, f'Straddle Price / VWAP / OIWAP — {title_base}', height=500, width=1400)
fig5.update_xaxes(tickangle=-45, nticks=20, row=2, col=1)
fig5.show()

In [ ]:
# ── Figure 6: OHLC Candlestick + Volume ───────────────────────────────────
fig6 = make_subplots(
    rows=2, cols=1, shared_xaxes=True,
    row_heights=[0.75, 0.25], vertical_spacing=0.03,
    subplot_titles=['OHLC Candlestick', 'Volume'],
)
fig6.add_trace(go.Candlestick(
    x=x, open=df['open'], high=df['high'], low=df['low'], close=df['close'],
    increasing_line_color=COLORS['candle_up'],
    decreasing_line_color=COLORS['candle_down'],
    name='OHLC',
), row=1, col=1)
if 'volume' in df.columns:
    vol_colors = [COLORS['candle_up'] if c >= o else COLORS['candle_down']
                  for c, o in zip(df['close'], df['open'])]
    fig6.add_trace(go.Bar(x=x, y=df['volume'], name='Volume',
        marker_color=vol_colors, opacity=0.6), row=2, col=1)
_add_day_boundaries(fig6, boundary_x, [1, 2], [1, 1])
_apply_dark_layout(fig6, f'OHLC Candlestick — {title_base}', height=600, width=1400)
fig6.update_xaxes(rangeslider_visible=False, row=1, col=1)
fig6.show()